# Vietnamese Medical NER / Assertion / Mapping — Kaggle runner

Chạy pipeline self-host (Qwen3-8B qua Ollama + ViHealthBERT/e5) trên 100 file `.txt`.

**Trước khi chạy — bật trong panel Settings bên phải:**
1. **Accelerator = GPU T4 x2** (hoặc P100)
2. **Internet = On** (bắt buộc: tải Ollama/model + gọi RxNorm API)
3. **Add Data** → thêm CẢ 2 dataset:
   - `data-input-viettel-race` — 100 file `.txt`
   - `ICD10-clean` — file `.xlsx` (ICD-10 đã làm sạch)

Chạy tuần tự từng cell. Khi chạy full 100 file, dùng **Save & Run All (Commit)** để job chạy độc lập.

## 1. Cấu hình — chỉnh cho khớp Dataset của bạn

In [ ]:
import os, glob
from collections import Counter

# 1) In cây /kaggle/input để thấy cấu trúc/tên mount THẬT
print("=== Nội dung /kaggle/input ===")
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count("/") - 2
        print("  " * depth + (os.path.basename(root) or root) + "/")
        for f in files[:5]:
            print("  " * (depth + 1) + f)
        if len(files) > 5:
            print("  " * (depth + 1) + f"... (+{len(files) - 5} file nữa)")
else:
    print("  (chưa có /kaggle/input — bạn chưa Add Data)")

# 2) Tự tìm INPUT_DIR = thư mục chứa NHIỀU .txt nhất (bất kể tên dataset)
_all_txt = glob.glob("/kaggle/input/**/*.txt", recursive=True)
INPUT_DIR = Counter(os.path.dirname(p) for p in _all_txt).most_common(1)[0][0] if _all_txt else None

# 3) Tự tìm ICD10_PATH = file .xlsx (ưu tiên tên chứa 'icd')
_all_xlsx = glob.glob("/kaggle/input/**/*.xlsx", recursive=True)
_icd_pref = [p for p in _all_xlsx if "icd" in os.path.basename(p).lower()]
ICD10_PATH = (_icd_pref or _all_xlsx or [None])[0]

REPO_URL = "https://github.com/jasmine95dn/vn-medical-ner-assertion.git"
REPO_DIR = "/kaggle/working/vn-medical-ner-assertion"

print("\nINPUT_DIR :", INPUT_DIR, "| số .txt:", len(_all_txt))
print("ICD10_PATH:", ICD10_PATH, "| tổng .xlsx thấy được:", len(_all_xlsx))
assert INPUT_DIR and glob.glob(f"{INPUT_DIR}/*.txt"), \
    "Không thấy .txt trong /kaggle/input — kiểm tra đã Add Data dataset 100 file test chưa (xem cây ở trên)"
assert ICD10_PATH and os.path.isfile(ICD10_PATH), \
    "Không thấy .xlsx trong /kaggle/input — kiểm tra đã Add Data dataset ICD-10 chưa (xem cây ở trên)"
print("\nOK — đường dẫn hợp lệ.")

## 2. Lấy code từ GitHub (clone lần đầu, các lần sau tự `git pull`)

In [ ]:
import os, subprocess
if os.path.isdir(REPO_DIR):
    print("repo đã có → git pull")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 3. Cài thư viện Python

In [ ]:
!pip install -q -r requirements.txt --break-system-packages

## 4. Cài & khởi động Ollama, tải model Qwen3-8B

Server chạy nền; lần đầu `ollama pull qwen3:8b` tải ~5GB nên hơi lâu.

In [ ]:
import subprocess, time, os

# Bản cài Ollama mới cần 'zstd' để giải nén — Kaggle chưa có sẵn, cài trước.
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd",
               shell=True, check=True)

# cài Ollama nếu chưa có
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# chạy server nền
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"])
time.sleep(8)  # chờ server sẵn sàng

subprocess.run(["ollama", "pull", "qwen3:8b"], check=True)
print("Ollama sẵn sàng.")

## 5. Smoke test — 3 file, bỏ candidate (nhanh, kiểm pipeline chạy được)

In [ ]:
!python main.py --input-dir "{INPUT_DIR}" --icd10-path "{ICD10_PATH}" --limit 3 --no-candidates

In [ ]:
# xem thử vài entity đầu ra
import json, glob
for f in sorted(glob.glob("output/*.json"))[:3]:
    print("===", f, "===")
    data = json.load(open(f, encoding="utf-8"))
    print(json.dumps(data[:5], ensure_ascii=False, indent=2))

## 6. Chấm điểm nhanh trên validation set (NER + assertion)

In [ ]:
!python evaluate.py --run --save-pred output/val_pred.json

## 7. Chạy FULL 100 file (có candidate mapping ICD-10 + RxNorm)

Lần đầu sẽ tải 2 model embedding (ViHealthBERT + e5-base) từ HuggingFace.
Nên chạy bằng **Save & Run All (Commit)** để không cần giữ tab mở.

In [ ]:
!python main.py --input-dir "{INPUT_DIR}" --icd10-path "{ICD10_PATH}"

## 8. Đóng gói output để tải về

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/output", "zip", "output")
print("Đã nén →/kaggle/working/output.zip (tải ở tab Output)")
import glob
print("Số file JSON:", len(glob.glob("output/*.json")))